# 분류

##### - pos_label
- 어떤 클래스를 Positive(양성) 로 볼지 지정하는 옵션
- `f1_score`, `precision_score`, `recall_score` 에서 사용
- `accuracy_score`, `roc_auc_score` 에선 사용 X

```python
f1_score(y_true, y_pred, pos_label="yes") → "yes" 를 Positive class 로 간주하여 계산

##### - roc_auc_score
- ROC 곡선 아래 면적(AUC)을 계산하는 평가지표
- ROC 곡선은 **분류 확률값의 변화**를 기반으로 생성됨
- 따라서 `predict()`가 아닌 `predict_proba()` 사용 권장
- rf.classes_에서 두번째(큰 값)을 positive로 삼음
- 다중 분류일 경우 `roc_auc_score(y_true, predict_proba, multi_class="ovr")`

##### - 분류(Classification)
- train / valid 데이터의 클래스 비율 유지를 위해 보통 `stratify = y` 사용

##### - predict_proba
- 각 클래스에 대한 확률을 보고 싶을 때 사용
- [:, 1] : 2번째 클래스에 해당하는 확률
- 다중 클래스에 대한 확률을 보고 싶을 때는 [:, 1] 이런거 하면 안됨

## 서비스 이탈예측 데이터
데이터 설명 : 고객의 신상정보 데이터를 통한 회사 서비스 이탈 예측 (종속변수 : Exited)  
x_train : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_train.csv  
y_train : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/y_train.csv  
x_test : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/y_test.csv  
데이터 출처 : https://www.kaggle.com/shubh0799/churn-modelling 에서 변형

In [1]:
import pandas as pd
#데이터 로드
x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/churnk/X_test.csv")

display(x_train.head())
display(y_train.head())

,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,15799217,Zetticci,791,Germany,Female,35,7,52436.20,1,1,0,161051.75
1,15748986,Bischof,705,Germany,Male,42,8,166685.92,2,1,1,55313.51
2,15722004,Hsiung,543,France,Female,31,4,138317.94,1,0,0,61843.73
3,15780966,Pritchard,709,France,Female,32,2,0.00,2,0,0,109681.29
4,15636731,Ts'ai,714,Germany,Female,36,1,101609.01,2,1,1,447.73


,CustomerId,Exited
0,15799217,0
1,15748986,0
2,15722004,0
3,15780966,0
4,15636731,0


In [2]:
drop_col = ["CustomerId", "Surname"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
display(x_train_drop, x_test_drop)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,791,Germany,Female,35,7,52436.20,1,1,0,161051.75
1,705,Germany,Male,42,8,166685.92,2,1,1,55313.51
2,543,France,Female,31,4,138317.94,1,0,0,61843.73
3,709,France,Female,32,2,0.00,2,0,0,109681.29
4,714,Germany,Female,36,1,101609.01,2,1,1,447.73
...,...,...,...,...,...,...,...,...,...,...
6494,696,Spain,Male,24,9,0.00,1,0,0,10883.52
6495,513,Germany,Male,34,7,60515.13,1,0,0,124571.09
6496,663,Spain,Female,22,9,0.00,1,1,0,29135.89
6497,635,Spain,Female,48,2,0.00,2,1,1,136551.25


,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,802,France,Female,60,3,92887.06,1,1,0,39473.63
1,602,France,Female,56,3,115895.22,3,1,0,4176.17
2,801,France,Female,32,4,75170.54,1,1,1,37898.50
3,693,Spain,Female,34,10,107556.06,2,0,0,154631.35
4,592,France,Female,62,5,0.00,1,1,1,100941.57
...,...,...,...,...,...,...,...,...,...,...
3496,496,Germany,Female,55,4,125292.53,1,1,1,31532.96
3497,556,Germany,Female,31,1,128663.81,2,1,0,125083.29
3498,589,France,Female,61,1,0.00,1,1,0,61108.56
3499,714,France,Female,25,4,0.00,2,0,0,82500.84


In [3]:
x_train_dummies = pd.get_dummies(x_train_drop)
y = y_train["Exited"]

x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns] # train과 컬럼 순서 동일하게 하기 (더미화 하면서 순서대로 정렬을 이미 하기 때문에 오류가 난다면 해당 컬럼이 누락된것)

display(x_train_dummies, x_test_dummies)


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_ male,Gender_Female,Gender_Male,Gender_female
0,791,35,7,52436.20,1,1,0,161051.75,False,True,False,False,True,False,False
1,705,42,8,166685.92,2,1,1,55313.51,False,True,False,False,False,True,False
2,543,31,4,138317.94,1,0,0,61843.73,True,False,False,False,True,False,False
3,709,32,2,0.00,2,0,0,109681.29,True,False,False,False,True,False,False
4,714,36,1,101609.01,2,1,1,447.73,False,True,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6494,696,24,9,0.00,1,0,0,10883.52,False,False,True,False,False,True,False
6495,513,34,7,60515.13,1,0,0,124571.09,False,True,False,False,False,True,False
6496,663,22,9,0.00,1,1,0,29135.89,False,False,True,False,True,False,False
6497,635,48,2,0.00,2,1,1,136551.25,False,False,True,False,True,False,False


,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain,Gender_ male,Gender_Female,Gender_Male,Gender_female
0,802,60,3,92887.06,1,1,0,39473.63,True,False,False,False,True,False,False
1,602,56,3,115895.22,3,1,0,4176.17,True,False,False,False,True,False,False
2,801,32,4,75170.54,1,1,1,37898.50,True,False,False,False,True,False,False
3,693,34,10,107556.06,2,0,0,154631.35,False,False,True,False,True,False,False
4,592,62,5,0.00,1,1,1,100941.57,True,False,False,False,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3496,496,55,4,125292.53,1,1,1,31532.96,False,True,False,False,True,False,False
3497,556,31,1,128663.81,2,1,0,125083.29,False,True,False,False,True,False,False
3498,589,61,1,0.00,1,1,0,61108.56,True,False,False,False,True,False,False
3499,714,25,4,0.00,2,0,0,82500.84,True,False,False,False,True,False,False


In [4]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)

print(len(X_train), len(X_valid), len(x_train_dummies))

4549 1950 6499


In [5]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(random_state= 42)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [6]:
predict_train_label = rf.predict(X_train) # 각각 모델의 최종 예측값 출력 -> 분류 모델 : 0,1,2,3 등 가장 확률 높은 클래스 출력
predict_train_proba = rf.predict_proba(X_train)[:,1] # 두 번째 클래스(즉, positive class) 에 대한 확률만 1차원 배열로 추출
print(predict_train_label, predict_train_proba)

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:,1]
print(predict_valid_label, predict_valid_proba)

[0 0 0 ... 1 0 0] [0.03 0.08 0.03 ... 0.85 0.22 0.13]
[0 0 0 ... 0 0 0] [0.05 0.08 0.02 ... 0.06 0.32 0.09]


In [7]:
from sklearn.metrics import accuracy_score , f1_score, recall_score, roc_auc_score ,precision_score
# 문제에서 묻는 것에 따라 모델 성능 확인하기
# 정확도 (accuracy) , f1_score , recall , precision -> model.predict로 결과뽑기
# auc , 확률이라는 표현있으면 model.predict_proba로 결과뽑고 첫번째 행의 값을 가져오기 model.predict_proba()[:,1]

print('train accuracy :', accuracy_score(Y_train, predict_train_label))
print('valida accuracy :', accuracy_score(Y_valid, predict_valid_label))
print('\n')

print('train f1_score :', f1_score(Y_train, predict_train_label))
print('valida f1_score :', f1_score(Y_valid, predict_valid_label))

print('\n')
print('train recall_score :', recall_score(Y_train, predict_train_label))
print('valida recall_score :', recall_score(Y_valid, predict_valid_label))

print('\n')
print('train precision_score :', precision_score(Y_train, predict_train_label))
print('valida precision_score :', precision_score(Y_valid, predict_valid_label))

print('\n')
print('train auc :', roc_auc_score(Y_train, predict_train_proba)) # roc_auc_score는 predict_proba로 0, 1만이 아니라 각 클래스에 대한 확률로 계산
print('valida auc :', roc_auc_score(Y_valid, predict_valid_proba))

train accuracy : 1.0
valida accuracy : 0.8630769230769231


train f1_score : 1.0
valida f1_score : 0.5572139303482587


train recall_score : 1.0
valida recall_score : 0.42317380352644834


train precision_score : 1.0
valida precision_score : 0.8155339805825242


train auc : 1.0
valida auc : 0.844464520607713


In [8]:
# test데이터 마찬가지 위와 같은 방식
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:,1]

In [9]:
data = pd.DataFrame({
    "CustomerId" : x_test["CustomerId"],
    "pred" : predict_test_label   # 문제 평가지표가 roc_auc_score이면 predict_test_proba
})

data

,CustomerId,pred
0,15601012,1
1,15734762,1
2,15586757,0
3,15590888,0
4,15726087,0
...,...,...
3496,15733966,0
3497,15669994,0
3498,15712403,1
3499,15643819,0


## 이직여부 판단 데이터
데이터 설명 : 이직여부 판단 데이터 (target: 1: 이직 , 0 : 이직 x)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/arashnic/hr-analytics-job-change-of-data-scientists (참고, 데이터 수정)

In [10]:
x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/HRdata/X_test.csv")

display(x_train.head())
display(y_train.head())

,enrollee_id,city,city_development_index,gender,relevent_experience,enrolled_university,education_level,major_discipline,experience,company_size,company_type,last_new_job,training_hours
0,25298,city_138,0.836,Male,No relevent experience,Full time course,High School,NaN,5,100-500,Pvt Ltd,1,45
1,4241,city_160,0.920,Male,No relevent experience,Full time course,High School,NaN,5,NaN,NaN,1,17
2,24086,city_57,0.866,Male,No relevent experience,no_enrollment,Graduate,STEM,10,NaN,NaN,1,50
3,26773,city_16,0.910,Male,Has relevent experience,no_enrollment,Graduate,STEM,>20,50-99,Pvt Ltd,>4,135
4,32325,city_143,0.740,NaN,No relevent experience,Full time course,Graduate,STEM,5,NaN,NaN,never,17


,enrollee_id,target
0,25298,0.0
1,4241,1.0
2,24086,0.0
3,26773,0.0
4,32325,1.0


In [11]:
drop_col = ["enrollee_id", "city"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["target"]

In [12]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

display(x_train_dummies, x_test_dummies)

,city_development_index,training_hours,gender_Female,gender_Male,gender_Other,relevent_experience_Has relevent experience,relevent_experience_No relevent experience,enrolled_university_Full time course,enrolled_university_Part time course,enrolled_university_no_enrollment,...,company_type_NGO,company_type_Other,company_type_Public Sector,company_type_Pvt Ltd,last_new_job_1,last_new_job_2,last_new_job_3,last_new_job_4,last_new_job_>4,last_new_job_never
0,0.836,45,False,True,False,False,True,True,False,False,...,False,False,False,True,True,False,False,False,False,False
1,0.920,17,False,True,False,False,True,True,False,False,...,False,False,False,False,True,False,False,False,False,False
2,0.866,50,False,True,False,False,True,False,False,True,...,False,False,False,False,True,False,False,False,False,False
3,0.910,135,False,True,False,True,False,False,False,True,...,False,False,False,True,False,False,False,False,True,False
4,0.740,17,False,False,False,False,True,True,False,False,...,False,False,False,False,False,False,False,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
12447,0.624,55,False,True,False,True,False,False,False,True,...,False,False,False,True,True,False,False,False,False,False
12448,0.920,7,False,True,False,True,False,False,False,True,...,False,False,False,True,False,False,False,False,True,False
12449,0.920,36,True,False,False,True,False,False,False,True,...,False,False,False,True,True,False,False,False,False,False
12450,0.920,41,False,True,False,True,False,True,False,False,...,False,False,False,False,False,True,False,False,False,False


,city_development_index,training_hours,gender_Female,gender_Male,gender_Other,relevent_experience_Has relevent experience,relevent_experience_No relevent experience,enrolled_university_Full time course,enrolled_university_Part time course,enrolled_university_no_enrollment,...,company_type_NGO,company_type_Other,company_type_Public Sector,company_type_Pvt Ltd,last_new_job_1,last_new_job_2,last_new_job_3,last_new_job_4,last_new_job_>4,last_new_job_never
0,0.899,23,False,True,False,False,True,False,False,True,...,False,False,False,False,True,False,False,False,False,False
1,0.725,39,False,True,False,False,True,False,True,False,...,False,False,False,False,False,False,False,False,False,True
2,0.920,262,False,True,False,False,True,True,False,False,...,False,False,False,False,False,True,False,False,False,False
3,0.896,78,False,True,False,False,True,True,False,False,...,False,False,False,False,False,False,False,False,False,True
4,0.689,125,True,False,False,True,False,True,False,False,...,False,False,False,True,True,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6701,0.920,8,False,True,False,True,False,False,True,False,...,False,False,False,True,False,True,False,False,False,False
6702,0.884,8,False,True,False,False,True,True,False,False,...,False,False,False,False,True,False,False,False,False,False
6703,0.836,20,False,True,False,False,True,False,True,False,...,False,False,False,True,False,False,False,True,False,False
6704,0.897,86,False,True,False,False,True,True,False,False,...,False,False,False,False,False,False,False,False,False,True


In [13]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

8716 3736 12452


In [14]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [15]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:,1]

In [16]:
from sklearn.metrics import accuracy_score , f1_score, recall_score, roc_auc_score ,precision_score
print('train accuracy :', accuracy_score(Y_train, predict_train_label))
print('valid accuracy :', accuracy_score(Y_valid, predict_valid_label))
print('\n')

print('train f1_score :', f1_score(Y_train, predict_train_label))
print('valid f1_score :', f1_score(Y_valid, predict_valid_label))
print('\n')

print('train recall_score :', recall_score(Y_train, predict_train_label))
print('valid recall_score :', recall_score(Y_valid, predict_valid_label))
print('\n')

print('train precision_score :', precision_score(Y_train, predict_train_label))
print('valid precision_score :', precision_score(Y_valid, predict_valid_label))
print('\n')

print('train auc :', roc_auc_score(Y_train, predict_train_proba))
print('valid auc :', roc_auc_score(Y_valid, predict_valid_proba))

train accuracy : 0.9988526847177605
valid accuracy : 0.7917558886509636


train f1_score : 0.9976979742173112
valid f1_score : 0.5044585987261146


train recall_score : 0.9972388403129314
valid recall_score : 0.4248927038626609


train precision_score : 0.9981575310916628
valid precision_score : 0.6206896551724138


train auc : 0.9999954986443533
valid auc : 0.7946933947824383


In [17]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [18]:
data = pd.DataFrame({
    "predict" : predict_test_label
})
    
data

,predict
0,1.0
1,0.0
2,0.0
3,0.0
4,0.0
...,...
6701,0.0
6702,0.0
6703,0.0
6704,0.0


## 정시 배송 여부 판단 (2회기출)
데이터 설명 : e-commerce 배송의 정시 도착여부 (1: 정시배송 0 : 정시미배송)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/prachi13/customer-analytics (참고, 데이터 수정)

In [19]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/shipping/X_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Warehouse_block,Mode_of_Shipment,Customer_care_calls,Customer_rating,Cost_of_the_Product,Prior_purchases,Product_importance,Gender,Discount_offered,Weight_in_gms
0,6045,A,Flight,4,3,266,5,high,F,5,1590
1,44,F,Ship,3,1,174,2,low,M,44,1556
2,7940,F,Road,4,1,154,10,high,M,10,5674
3,1596,F,Ship,4,3,158,3,medium,F,27,1207
4,4395,A,Flight,5,3,175,3,low,M,7,4833


,ID,Reached.on.Time_Y.N
0,6045,0
1,44,1
2,7940,1
3,1596,1
4,4395,1


In [20]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y =y_train["Reached.on.Time_Y.N"]

In [21]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [22]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)

In [23]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [24]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]

In [25]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score :", f1_score(Y_train, predict_train_label))
print("valid f1_score :", f1_score(Y_valid, predict_valid_label))
print("\n")

print("train recall_score :", recall_score(Y_train, predict_train_label))
print("valid recall_score :", recall_score(Y_valid, predict_valid_label))
print("\n")

print("train precision_score :", precision_score(Y_train, predict_train_label))
print("valid precision_score :", precision_score(Y_valid, predict_valid_label))
print("\n")

print("train roc_auc_score :", roc_auc_score(Y_train, predict_train_proba))
print("valid roc_auc_score :", roc_auc_score(Y_valid, predict_valid_proba))

train accuracy_score : 1.0
valid accuracy_score : 0.6651515151515152


train f1_score : 1.0
valid f1_score : 0.7041499330655957


train recall_score : 1.0
valid recall_score : 0.668077900084674


train precision_score : 1.0
valid precision_score : 0.7443396226415094


train roc_auc_score : 1.0
valid roc_auc_score : 0.7450046046126669


In [26]:
predict_test_label = rf.predict(x_test_dummies)
preidct_test_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [27]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test_label
})

data

,ID,predict
0,6811,1
1,4320,0
2,5732,1
3,7429,1
4,2191,1
...,...,...
4396,2610,1
4397,3406,1
4398,10395,0
4399,3646,0


## 성인 건강검진 데이터
데이터 설명 : 2018년도 성인의 건강검 진데이터 (종속변수 : 흡연상태 1- 흡연, 0-비흡연 )  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_test.csv x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/y_test.csv  
데이터 출처 :https://www.data.go.kr/data/15007122/fileData.do (참고, 데이터 수정)

In [28]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/smoke/x_test.csv")


display(x_train.head())
display(y_train.head())

,ID,성별코드,연령대코드(5세단위),신장(5Cm단위),체중(5Kg단위),허리둘레,시력(좌),시력(우),청력(좌),청력(우),...,LDL콜레스테롤,혈색소,요단백,혈청크레아티닌,(혈청지오티)AST,(혈청지오티)ALT,감마지티피,구강검진수검여부,치아우식증유무,치석
0,0,F,40,155,60,81.3,1.2,1.0,1.0,1.0,...,126.0,12.9,1.0,0.7,18.0,19.0,27.0,Y,0.0,Y
1,1,F,40,160,60,81.0,0.8,0.6,1.0,1.0,...,127.0,12.7,1.0,0.6,22.0,19.0,18.0,Y,0.0,Y
2,2,M,55,170,60,80.0,0.8,0.8,1.0,1.0,...,151.0,15.8,1.0,1.0,21.0,16.0,22.0,Y,0.0,N
3,3,M,40,165,70,88.0,1.5,1.5,1.0,1.0,...,226.0,14.7,1.0,1.0,19.0,26.0,18.0,Y,0.0,Y
4,4,F,40,155,60,86.0,1.0,1.0,1.0,1.0,...,107.0,12.5,1.0,0.6,16.0,14.0,22.0,Y,0.0,N


,ID,흡연상태
0,0,0
1,1,0
2,2,1
3,3,0
4,4,0


In [29]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["흡연상태"]

In [30]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [31]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
# stratify = y : 훈련 데이터와 검증 데이터가 원래 데이터의 클래스 비율을 그대로 유지하도록 층화추출
print(len(X_train), len(X_valid), len(x_train_dummies))

31187 13366 44553


In [32]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [33]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]



In [34]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score :", f1_score(Y_train, predict_train_label))
print("valid f1_score :", f1_score(Y_valid, predict_valid_label))
print("\n")

print("train recall_score :", recall_score(Y_train, predict_train_label))
print("valid recall_score :", recall_score(Y_valid, predict_valid_label))
print("\n")

print("train precision_score :", precision_score(Y_train, predict_train_label))
print("valid precision_score :", precision_score(Y_valid, predict_valid_label))
print("\n")

print("train roc_auc_score :", roc_auc_score(Y_train, predict_train_proba))
print("valid roc_auc_score :", roc_auc_score(Y_valid, predict_valid_proba))
print("\n")

train accuracy_score : 1.0
valid accuracy_score : 0.7554990273829119


train f1_score : 1.0
valid f1_score : 0.6791044776119403


train recall_score : 1.0
valid recall_score : 0.7048512026090501


train precision_score : 1.0
valid precision_score : 0.6551724137931034


train roc_auc_score : 1.0
valid roc_auc_score : 0.8369235359992444




In [35]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [36]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test_label
})

data

,ID,predict
0,8,0
1,17,0
2,20,1
3,24,0
4,25,0
...,...,...
11134,55676,0
11135,55681,0
11136,55683,0
11137,55684,0


## 자동차 보험가입 예측데이터
데이터 설명 : 자동차 보험 가입 예측 (종속변수 Response: 1 : 가입 , 0 :미가입)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/y_test.csv  
데이터 출처 :https://www.kaggle.com/anmolkumar/health-insurance-cross-sell-prediction(참고, 데이터 수정)근처 자동차 대리점

In [37]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/insurance/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Gender,Age,Driving_License,Region_Code,Previously_Insured,Vehicle_Age,Vehicle_Damage,Annual_Premium,Policy_Sales_Channel,Vintage,id
0,0,Female,23,1,8.0,0,< 1 Year,Yes,61354.0,152.0,235,NaN
1,1,Male,27,1,28.0,1,< 1 Year,No,38036.0,152.0,207,NaN
2,2,Female,23,1,45.0,0,< 1 Year,Yes,25984.0,152.0,217,NaN
3,3,Male,22,1,46.0,0,< 1 Year,No,39499.0,152.0,277,NaN
4,4,Male,32,1,30.0,1,< 1 Year,No,38771.0,152.0,251,NaN


,ID,Response
0,0,0
1,1,0
2,2,1
3,3,0
4,4,0


In [38]:
drop_col = ["ID", "id"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Response"]

In [39]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [40]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

213420 91467 304887


In [41]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [42]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]

In [43]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score ,roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score :", f1_score(Y_train, predict_train_label))
print("valid f1_score :", f1_score(Y_valid, predict_valid_label))
print("\n")

print("train recall_score :", recall_score(Y_train, predict_train_label))
print("valid recall_score :", recall_score(Y_valid, predict_valid_label))
print("\n")

print("train precision_score :", precision_score(Y_train, predict_train_label))
print("valid precision_score :", precision_score(Y_valid, predict_valid_label))
print("\n")

print("train roc_auc_score :", roc_auc_score(Y_train, predict_train_proba))
print("valid roc_auc_score :", roc_auc_score(Y_valid, predict_valid_proba))

train accuracy_score : 0.99993440164933
valid accuracy_score : 0.8658313927427378


train f1_score : 0.9997323340471093
valid f1_score : 0.18219378915100626


train recall_score : 0.9995412317926368
valid recall_score : 0.12193381500312193


train precision_score : 0.9999235093892225
valid precision_score : 0.36021080368906455


train roc_auc_score : 0.9999999643749466
valid roc_auc_score : 0.8321449382447135


In [44]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [45]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test_label
})

data

,ID,predict
0,17,0
1,19,0
2,21,0
3,23,0
4,32,0
...,...,...
76217,381091,0
76218,381097,0
76219,381102,0
76220,381105,1


## ❌ 비행탑승 경험 만족도 데이터
데이터 설명 : 비행탑승 경험 만족도 (satisfaction 컬럼 : ‘neutral or dissatisfied’ or satisfied ) (83123, 24) shape  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/y_test.csv  
데이터 출처 :https://www.kaggle.com/teejmahal20/airline-passenger-satisfaction?select=train.csv (참고, 데이터 수정)

### test 데이터에 대해서 neutral or dissatisfied라고 예측할 확률을 구하고 그 확률 값을 제출하라

In [46]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/airline/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Gender,Customer Type,Age,Type of Travel,Class,Flight Distance,Inflight wifi service,Departure/Arrival time convenient,Ease of Online booking,...,Inflight entertainment,On-board service,Leg room service,Baggage handling,Checkin service,Inflight service,Cleanliness,Departure Delay in Minutes,Arrival Delay in Minutes,id
0,0,Female,Loyal Customer,54,Personal Travel,Eco,1068,3,4,3,...,5,5,3,5,3,5,3,47,22.0,NaN
1,2,Male,Loyal Customer,20,Personal Travel,Eco,1546,4,4,4,...,4,3,3,4,4,4,4,5,2.0,NaN
2,3,Male,Loyal Customer,59,Business travel,Business,2962,0,4,0,...,1,1,1,1,5,1,4,54,46.0,NaN
3,4,Male,Loyal Customer,35,Business travel,Eco Plus,106,5,4,4,...,5,2,1,5,4,4,5,130,121.0,NaN
4,5,Female,Loyal Customer,9,Business travel,Business,2917,3,3,3,...,4,4,4,5,4,3,4,0,0.0,NaN


,ID,satisfaction
0,0,neutral or dissatisfied
1,2,neutral or dissatisfied
2,3,satisfied
3,4,satisfied
4,5,satisfied


In [47]:
drop_col = ["ID", "id"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["satisfaction"]

In [48]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [49]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

58186 24937 83123


In [50]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [51]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]

In [52]:
# ❌ 
from sklearn.metrics import accuracy_score, f1_score, recall_score, roc_auc_score, precision_score

print("train accuracy_score : ", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score : ", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score : ", f1_score(Y_train, predict_train_label, pos_label = "neutral or dissatisfied")) # pos_label : f1_score에서 양성이라고 할 항목 지정
print("valid f1_score : ", f1_score(Y_valid, predict_valid_label, pos_label = "neutral or dissatisfied"))
print("\n")

print("train recall_score : ", recall_score(Y_train, predict_train_label, pos_label = "neutral or dissatisfied"))
print("valid recall_score : ", recall_score(Y_valid, predict_valid_label, pos_label = "neutral or dissatisfied"))
print("\n")

print("train roc_auc_score : ", roc_auc_score(Y_train, predict_train_proba)) # roc_auc_score는 y.unique()의 뒤 클래스를 기준으로 생각
print("valid roc_auc_score : ", roc_auc_score(Y_valid, predict_valid_proba))
print("\n")

print("train precision_score : ", precision_score(Y_train, predict_train_label, pos_label = "neutral or dissatisfied"))
print("valid precision_score : ", precision_score(Y_valid, predict_valid_label, pos_label = "neutral or dissatisfied"))

train accuracy_score :  1.0
valid accuracy_score :  0.9613024822552834


train f1_score :  1.0
valid f1_score :  0.9662481200377742


train recall_score :  1.0
valid recall_score :  0.9774962847639941


train roc_auc_score :  1.0
valid roc_auc_score :  0.9938208083943332


train precision_score :  1.0
valid precision_score :  0.955255878284924


In [53]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:, 0] # 0이 "neutral or dissatisfied"
###############
# y.unique() 하면 ["neutral or dissatisfied", "satisfied"]이므로 0이 "neutral or dissatisfied"

In [54]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test_proba
})

data

,ID,predict
0,1,0.84
1,16,0.92
2,17,1.00
3,25,1.00
4,27,0.20
...,...,...
20776,103895,1.00
20777,103896,1.00
20778,103897,0.00
20779,103900,1.00


## ❌ 수질 음용성 여부 데이터
데이터 설명 : 수질 음용성 여부 (Potablillity 컬럼 : 0 ,1 )  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/y_test.csv  
데이터 출처 :https://www.kaggle.com/adityakadiwal/water-potability

In [55]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/waters/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,ph,Hardness,Solids,Chloramines,Sulfate,Conductivity,Organic_carbon,Trihalomethanes,Turbidity
0,0,8.662710,173.531947,20333.079495,5.636388,439.787938,459.633120,16.283311,89.924253,5.120103
1,1,NaN,226.270824,15380.124079,6.661474,NaN,392.558205,14.083110,50.286395,4.516870
2,2,7.583770,217.283262,36343.407055,8.532726,375.964391,393.877683,17.442301,77.722257,3.642289
3,3,6.584813,182.375456,24723.106296,6.238920,NaN,414.350751,17.582615,78.213738,4.404132
4,4,7.179864,180.854211,10859.553752,8.263503,341.302486,358.056264,12.065317,83.329918,3.878447


,ID,Potability
0,0,0
1,1,1
2,2,0
3,3,0
4,4,0


In [56]:
print(x_train.isnull().sum(), "\n")
print(x_test.isnull().sum())

ID                   0
ph                 395
Hardness             0
Solids               0
Chloramines          0
Sulfate            617
Conductivity         0
Organic_carbon       0
Trihalomethanes    132
Turbidity            0
dtype: int64 

ID                   0
ph                 107
Hardness             0
Solids               0
Chloramines          0
Sulfate            152
Conductivity         0
Organic_carbon       0
Trihalomethanes     38
Turbidity            0
dtype: int64


In [57]:
# ❌ 
for col in x_train.columns:
    if x_train[col].isnull().any():

        mean_value = x_train[col].mean()

        x_train[col] = x_train[col].fillna(mean_value)
        x_test[col] = x_test[col].fillna(mean_value) # data leakage 때문에 test의 null을 train의 평균으로 대체

In [58]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Potability"]

In [59]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [60]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

1834 786 2620


In [61]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [62]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]

In [63]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score :", f1_score(Y_train, predict_train_label))
print("valid f1_score :", f1_score(Y_valid, predict_valid_label))
print("\n")

print("train recall_score :", recall_score(Y_train, predict_train_label))
print("valid recall_score :", recall_score(Y_valid, predict_valid_label))
print("\n")

print("train precision_score :", precision_score(Y_train, predict_train_label))
print("valid precision_score :", precision_score(Y_valid, predict_valid_label))
print("\n")

print("train roc_auc_score :", roc_auc_score(Y_train, predict_train_proba))
print("valid roc_auc_score :", roc_auc_score(Y_valid, predict_valid_proba))

train accuracy_score : 1.0
valid accuracy_score : 0.6564885496183206


train f1_score : 1.0
valid f1_score : 0.4230769230769231


train recall_score : 1.0
valid recall_score : 0.32247557003257327


train precision_score : 1.0
valid precision_score : 0.6149068322981367


train roc_auc_score : 1.0
valid roc_auc_score : 0.6355463676361583


In [64]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [65]:
data = pd.DataFrame({
    "predict" : predict_test_label
})

data

,predict
0,0
1,1
2,1
3,0
4,0
...,...
651,0
652,0
653,1
654,1


## ❌ 약물 분류 데이터
데이터 설명 : 투약하는 약을 분류 (종속변수 :Drug)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/y_test.csv  
데이터 출처 :https://www.kaggle.com/prathamtripathi/drug-classification(참고, 데이터 수정)

In [66]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/drug/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Age,Sex,BP,Cholesterol,Na_to_K
0,0,36,F,NORMAL,HIGH,16.753
1,1,47,F,LOW,HIGH,11.767
2,2,69,F,NORMAL,HIGH,10.065
3,3,35,M,LOW,NORMAL,9.170
4,4,49,M,LOW,NORMAL,11.014


,ID,Drug
0,0,0
1,1,3
2,2,4
3,3,4
4,4,4


In [67]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Drug"]

In [68]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [69]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

109 48 157


In [70]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [71]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train) #  # [:, 1]을 안하는 이유 : 다중 클래스에 대한 proba를 구해야 하기 때문에

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)

In [72]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score : ", f1_score(Y_train, predict_train_label, average = "macro")) # average : f1을 구할 때 다중 클래스에서 각 클래스의 F1을 구해 단순 평균
print("validation f1_score : ", f1_score(Y_valid, predict_valid_label, average = "macro"))
print("\n")

print("train recall_score : ", recall_score(Y_train, predict_train_label, average = "macro"))
print("validation recall_score : ", recall_score(Y_valid, predict_valid_label, average = "macro"))
print("\n")

print("train roc_auc_score : ", roc_auc_score(Y_train, predict_train_proba, multi_class='ovr')) # multi_class : 다중 클래스에서 roc_auc_score를 구할 떄 각 클래스를 positive vs 나머지 전체로 비교
print("validation roc_auc_score : ", roc_auc_score(Y_valid, predict_valid_proba, multi_class='ovr'))
print("\n")

print("train precision_score : ", precision_score(Y_train, predict_train_label, average = "macro"))
print("validation precision_score : ", precision_score(Y_valid, predict_valid_label, average = "macro"))

train accuracy_score : 1.0
valid accuracy_score : 0.9791666666666666


train f1_score :  1.0
validation f1_score :  0.9532467532467532


train recall_score :  1.0
validation recall_score :  0.95


train roc_auc_score :  1.0
validation roc_auc_score :  0.9938002114164904


train precision_score :  1.0
validation precision_score :  0.9666666666666668


In [73]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)

In [74]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test_label
})

data

,ID,predict
0,8,0
1,9,4
2,14,1
3,25,4
4,26,0
5,27,4
6,41,4
7,50,4
8,53,0
9,56,1


## ❌ 사기회사 분류 데이터
데이터 설명 : 사기회사 분류 (종속변수 : Risk 1: 사기 , 0 : 정상)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/y_test.csv  
데이터 출처 :https://www.kaggle.com/sid321axn/audit-data(참고, 데이터 수정)

In [75]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/audit/x_test.csv")

display(x_train.head())
display(y_train.head())


,ID,Sector_score,LOCATION_ID,PARA_A,Score_A,Risk_A,PARA_B,Score_B,Risk_B,TOTAL,...,PROB,RiSk_E,History,Prob,Risk_F,Score,Inherent_Risk,CONTROL_RISK,Detection_Risk,Audit_Risk
0,0,2.37,16,0.01,0.2,0.002,0.007,0.2,0.0014,0.017,...,0.2,0.4,0,0.2,0.0,2.0,1.4034,0.4,0.5,0.28068
1,2,55.57,9,1.06,0.4,0.424,0.000,0.2,0.0000,1.060,...,0.2,0.4,0,0.2,0.0,2.2,1.8240,0.4,0.5,0.36480
2,3,55.57,16,2.42,0.6,1.452,3.530,0.6,2.1180,5.950,...,0.2,0.4,0,0.2,0.0,3.8,7.4940,0.4,0.5,1.49880
3,4,2.37,9,0.31,0.2,0.062,0.690,0.2,0.1380,1.000,...,0.2,0.4,0,0.2,0.0,2.0,1.6000,0.4,0.5,0.32000
4,5,55.57,6,0.62,0.2,0.124,0.420,0.2,0.0840,1.040,...,0.2,0.4,0,0.2,0.0,2.0,1.6080,0.4,0.5,0.32160


,ID,Risk
0,0,0
1,2,0
2,3,1
3,4,0
4,5,0


In [76]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Risk"]

In [77]:
# ❌ 
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [78]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

434 186 620


In [79]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [80]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]

In [81]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score : ", f1_score(Y_train, predict_train_label))
print("validation f1_score : ", f1_score(Y_valid, predict_valid_label))
print("\n")

print("train recall_score : ", recall_score(Y_train, predict_train_label))
print("validation recall_score : ", recall_score(Y_valid, predict_valid_label))
print("\n")

print("train roc_auc_score : ", roc_auc_score(Y_train, predict_train_proba))
print("validation roc_auc_score : ", roc_auc_score(Y_valid, predict_valid_proba))
print("\n")

print("train precision_score : ", precision_score(Y_train, predict_train_label))
print("validation precision_score : ", precision_score(Y_valid, predict_valid_label))

train accuracy_score : 1.0
valid accuracy_score : 1.0


train f1_score :  1.0
validation f1_score :  1.0


train recall_score :  1.0
validation recall_score :  1.0


train roc_auc_score :  1.0
validation roc_auc_score :  1.0


train precision_score :  1.0
validation precision_score :  1.0


In [82]:
predict_valid_label = rf.predict(x_test_dummies)
predict_valid_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [83]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_valid_label
})

data

,ID,predict
0,1,1
1,9,1
2,10,0
3,11,0
4,12,0
...,...,...
151,759,1
152,762,1
153,763,1
154,765,0


## 센서데이터 동작유형 분류 데이터
데이터 설명 : 센서데이터로 동작 유형 분류 (종속변수 pose : 0 ,1 구분)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/y_test.csv  
데이터 출처 :https://www.kaggle.com/kyr7plus/emg-4(참고, 데이터 수정)

In [84]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/muscle/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,motion_0,motion_1,motion_2,motion_3,motion_4,motion_5,motion_6,motion_7,motion_8,...,motion_54,motion_55,motion_56,motion_57,motion_58,motion_59,motion_60,motion_61,motion_62,motion_63
0,0,1.0,-2.0,-1.0,4.0,-5.0,-4.0,1.0,0.0,-15.0,...,0.0,-1.0,-13.0,-3.0,1.0,-1.0,-32.0,-22.0,-2.0,-3.0
1,2,20.0,0.0,0.0,1.0,5.0,6.0,-52.0,18.0,15.0,...,-70.0,-55.0,-38.0,-14.0,-12.0,-8.0,-34.0,-63.0,-87.0,-77.0
2,4,1.0,-1.0,1.0,4.0,-5.0,-8.0,1.0,-3.0,-14.0,...,1.0,12.0,-25.0,0.0,0.0,3.0,2.0,-27.0,1.0,0.0
3,5,13.0,2.0,1.0,-3.0,1.0,3.0,28.0,3.0,12.0,...,0.0,-21.0,-17.0,-2.0,0.0,-4.0,-17.0,-21.0,-21.0,25.0
4,6,-2.0,-7.0,-4.0,-8.0,16.0,44.0,1.0,3.0,-16.0,...,-1.0,2.0,-1.0,1.0,4.0,4.0,-17.0,-38.0,-3.0,3.0


,ID,pose
0,0,1
1,2,0
2,4,1
3,5,0
4,6,1


In [85]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["pose"]

In [86]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [87]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3 , random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

3255 1395 4650


In [88]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [89]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]

In [90]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score :", f1_score(Y_train, predict_train_label))
print("valid f1_score :", f1_score(Y_valid, predict_valid_label))
print("\n")

print("train recall_score :", recall_score(Y_train, predict_train_label))
print("valid recall_score :", recall_score(Y_valid, predict_valid_label))
print("\n")

print("train precision_score :", precision_score(Y_train, predict_train_label))
print("valid precision_score :", precision_score(Y_valid, predict_valid_label))
print("\n")

print("train roc_auc_score :", roc_auc_score(Y_train, predict_train_proba))
print("valid roc_auc_score :", roc_auc_score(Y_valid, predict_valid_proba))

train accuracy_score : 1.0
valid accuracy_score : 0.992831541218638


train f1_score : 1.0
valid f1_score : 0.9927745664739884


train recall_score : 1.0
valid recall_score : 0.9856527977044476


train precision_score : 1.0
valid precision_score : 1.0


train roc_auc_score : 1.0
valid roc_auc_score : 0.9999630014840516


In [91]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [92]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test_label
})

data

,ID,predict
0,1,1
1,3,1
2,8,1
3,10,1
4,17,0
...,...,...
1158,5786,0
1159,5796,0
1160,5802,1
1161,5811,1


## 당뇨여부판단 데이터
데이터 설명 : 당뇨여부 판단하기 (종속변수 Outcome : 1 당뇨 , 0 :정상)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/y_test.csv  
데이터 출처 :https://www.kaggle.com/pritsheta/diabetes-dataset(참고, 데이터 수정)

In [93]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/diabetes/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,0,8,126,88,36,108,38.5,0.349,49
1,1,0,74,52,10,36,27.8,0.269,22
2,2,1,140,74,26,180,24.1,0.828,23
3,3,6,162,62,0,0,24.3,0.178,50
4,4,2,94,68,18,76,26.0,0.561,21


,ID,Outcome
0,0,0
1,1,0
2,2,0
3,3,1
4,4,0


In [94]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Outcome"]

In [95]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [96]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42, stratify = y)
print(len(X_train), len(X_valid), len(x_train_dummies))

429 185 614


In [97]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'gini'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,'sqrt'
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [98]:
predict_train_label = rf.predict(X_train)
predict_train_proba = rf.predict_proba(X_train)[:, 1]

predict_valid_label = rf.predict(X_valid)
predict_valid_proba = rf.predict_proba(X_valid)[:, 1]

In [99]:
from sklearn.metrics import accuracy_score, f1_score, recall_score, precision_score, roc_auc_score
print("train accuracy_score :", accuracy_score(Y_train, predict_train_label))
print("valid accuracy_score :", accuracy_score(Y_valid, predict_valid_label))
print("\n")

print("train f1_score :", f1_score(Y_train, predict_train_label))
print("valid f1_score :", f1_score(Y_valid, predict_valid_label))
print("\n")

print("train recall_score :", recall_score(Y_train, predict_train_label))
print("valid recall_score :", recall_score(Y_valid, predict_valid_label))
print("\n")

print("train precision_score :", precision_score(Y_train, predict_train_label))
print("valid precision_score :", precision_score(Y_valid, predict_valid_label))
print("\n")

print("train roc_auc_score :", roc_auc_score(Y_train, predict_train_proba))
print("valid roc_auc_score :", roc_auc_score(Y_valid, predict_valid_proba))

train accuracy_score : 1.0
valid accuracy_score : 0.7891891891891892


train f1_score : 1.0
valid f1_score : 0.6976744186046512


train recall_score : 1.0
valid recall_score : 0.703125


train precision_score : 1.0
valid precision_score : 0.6923076923076923


train roc_auc_score : 1.0
valid roc_auc_score : 0.8527892561983471


In [100]:
predict_test_label = rf.predict(x_test_dummies)
predict_test_proba = rf.predict_proba(x_test_dummies)[:, 1]

In [101]:
rf.classes_

array([0, 1])

In [102]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test_label
})

data

,ID,predict
0,13,0
1,18,0
2,29,0
3,33,1
4,34,0
...,...,...
149,751,0
150,752,0
151,759,0
152,765,1


# 회귀

## 학생성적 예측 데이터
데이터 설명 : 학생성적 예측 (종속변수 :G3)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/ishandutta/student-performance-data-set (참고, 데이터 수정)

In [103]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/studentscore/X_test.csv")

display(x_train.head())
display(y_train.head())

,StudentID,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,...,romantic,famrel,freetime,goout,Dalc,Walc,health,absences,G1,G2
0,1714,GP,F,18,U,GT3,T,4,3,other,...,no,4,3,3,1,1,3,0,14,13
1,1254,GP,F,17,U,GT3,T,4,3,health,...,yes,4,4,3,1,3,4,0,13,15
2,1639,GP,F,16,R,GT3,T,4,4,health,...,no,2,4,4,2,3,4,6,10,11
3,1118,GP,M,16,U,GT3,T,4,4,services,...,no,5,3,3,1,3,5,0,15,13
4,1499,GP,M,19,U,GT3,T,3,2,services,...,yes,4,5,4,1,1,4,0,5,0


,StudentID,G3
0,1714,14
1,1254,15
2,1639,11
3,1118,13
4,1499,0


In [104]:
drop_col = ["StudentID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["G3"]

In [105]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [106]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)
print(len(X_train), len(X_valid), len(x_train_dummies))

474 204 678


In [107]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [108]:
predict_train = rf.predict(X_train)
predict_valid = rf.predict(X_valid)

In [109]:
import numpy as np
from sklearn.metrics import mean_squared_error ,mean_absolute_error ,mean_absolute_percentage_error, r2_score
print("train mean_squared_error :", mean_squared_error(Y_train, predict_train))
print("valid mean_squared_error :", mean_squared_error(Y_valid, predict_valid))
print("\n")

print('train rmse : ', np.sqrt(mean_squared_error(Y_train, predict_train)))
print('validation rmse : ', np.sqrt(mean_squared_error(Y_valid, predict_valid)))
print("\n")

print("train mean_absolute_error :", mean_absolute_error(Y_train, predict_train))
print("valid mean_absolute_error :", mean_absolute_error(Y_valid, predict_valid))
print("\n")

print("train mean_absolute_percentage_error :", mean_absolute_percentage_error(Y_train, predict_train))
print("valid mean_absolute_percentage_error :", mean_absolute_percentage_error(Y_valid, predict_valid))
print("\n")

print("train r2_score :", r2_score(Y_train, predict_train))
print("valid r2_score :", r2_score(Y_valid, predict_valid))

train mean_squared_error : 0.2784537974683544
valid mean_squared_error : 2.415214215686275


train rmse :  0.5276872155627369
validation rmse :  1.5540959480309686


train mean_absolute_error : 0.3409071729957806
valid mean_absolute_error : 0.9977941176470588


train mean_absolute_percentage_error : 237246587965066.03
valid mean_absolute_percentage_error : 539548896533994.8


train r2_score : 0.9812481010677709
valid r2_score : 0.8297438747872043


In [110]:
predict_test = rf.predict(x_test_dummies)

In [111]:
data = pd.DataFrame({
    "StudentID" : x_test["StudentID"],
    "predict" : predict_test
})

data

,StudentID,predict
0,1000,16.08
1,1008,11.22
2,1013,14.23
3,1014,6.86
4,1017,10.64
...,...,...
361,2032,15.66
362,2034,12.21
363,2035,15.94
364,2036,1.08


## ⚠️ 중고차 가격 예측 데이터
데이터 설명 : 중고차 가격 예측 데이터 (종속변수 :G3)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/y_test.csv  
데이터 출처 :https://www.kaggle.com/datasets/adityadesai13/used-car-dataset-ford-and-mercedes?select=vw.csv (참고, 데이터 수정)

In [112]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/carsprice/X_test.csv")

display(x_train.head())
display(y_train.head())

,carID,brand,model,year,transmission,mileage,fuelType,tax,mpg,engineSize
0,13207,hyundi,Santa Fe,2019,Semi-Auto,4223,Diesel,145.0,39.8,2.2
1,17314,vauxhall,GTC,2015,Manual,47870,Diesel,125.0,60.1,2.0
2,12342,audi,RS4,2019,Automatic,5151,Petrol,145.0,29.1,2.9
3,13426,vw,Scirocco,2016,Automatic,20423,Diesel,30.0,57.6,2.0
4,16004,skoda,Scala,2020,Semi-Auto,3569,Petrol,145.0,47.1,1.0


,carID,price
0,13207,31995
1,17314,7700
2,12342,58990
3,13426,12999
4,16004,16990


In [113]:
drop_col = ["carID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["price"]

In [114]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [115]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)
print(len(X_train), len(X_valid), len(x_train_dummies))

3472 1488 4960


In [116]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [117]:
predict_train = rf.predict(X_train)
predict_valid = rf.predict(X_valid)

In [118]:
from sklearn.metrics import mean_squared_error, mean_absolute_error ,mean_absolute_percentage_error, r2_score
print("train mse :", mean_squared_error(Y_train, predict_train))
print("valie mse :", mean_squared_error(Y_valid, predict_valid))
print("\n")

print("train rmse :", np.sqrt(mean_squared_error(Y_train, predict_train)))
print("valie rmse :", np.sqrt(mean_squared_error(Y_valid, predict_valid)))
print("\n")

print("train mae :", mean_absolute_error(Y_train, predict_train))
print("valie mae :", mean_absolute_error(Y_valid, predict_valid))
print("\n")


print("train mape :", mean_absolute_percentage_error(Y_train, predict_train))
print("valie mape :", mean_absolute_percentage_error(Y_valid, predict_valid))
print("\n")

print("train r2_score :", r2_score(Y_train, predict_train))
print("valie r2_score :", r2_score(Y_valid, predict_valid))

train mse : 2154713.5647287774
valie mse : 12399084.636438273


train rmse : 1467.8942621077233
valie rmse : 3521.2333970411946


train mae : 762.6361224784879
valie mae : 2059.1474022331877


train mape : 0.03683219075104364
valie mape : 0.09827838894058583


train r2_score : 0.9919148293694553
valie r2_score : 0.95490323340537


In [119]:
predict_test = rf.predict(x_test_dummies)

In [120]:
data = pd.DataFrame({
    "carID" : x_test["carID"],
    "predict" : predict_test
})

data

,carID,predict
0,12000,41935.06
1,12001,23849.61
2,12004,63980.54
3,12013,14948.32
4,12017,51294.28
...,...,...
2667,19618,47679.52
2668,19620,18654.54
2669,19626,20848.14
2670,19630,20153.13


## 의료 비용 예측 데이터
데이터 설명 : 의료비용 예측문제 (종속변수 :charges)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/y_test.csv  
데이터 출처 :https://www.kaggle.com/mirichoi0218/insurance/code(참고, 데이터 수정)

In [121]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/MedicalCost/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,age,sex,bmi,children,smoker,region
0,2,35,female,35.860,2,no,southeast
1,3,28,female,23.845,2,no,northwest
2,4,23,female,32.780,2,yes,southeast
3,6,52,female,25.300,2,yes,southeast
4,7,63,male,39.800,3,no,southwest


,ID,charges
0,2,5836.52040
1,3,4719.73655
2,4,36021.01120
3,6,24667.41900
4,7,15170.06900


In [122]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["charges"]

In [123]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [124]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)
print(len(X_train), len(X_valid), len(x_train_dummies))

749 321 1070


In [125]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [126]:
predict_train = rf.predict(X_train)
predict_valid = rf.predict(X_valid)

In [127]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
print("train mse :", mean_squared_error(Y_train, predict_train))
print("valid mse :", mean_squared_error(Y_valid, predict_valid))
print("\n")

print("train rmse :", np.sqrt(mean_squared_error(Y_train, predict_train)))
print("valid rmse :", np.sqrt(mean_squared_error(Y_valid, predict_valid)))
print("\n")

print("train mae :", mean_absolute_error(Y_train, predict_train))
print("valid mae :", mean_absolute_error(Y_valid, predict_valid))
print("\n")

print("train mape :", mean_absolute_percentage_error(Y_train, predict_train))
print("valid mape :", mean_absolute_percentage_error(Y_valid, predict_valid))
print("\n")

print("train r2_score :", r2_score(Y_train, predict_train))
print("valid r2_score :", r2_score(Y_valid, predict_valid))

train mse : 3269929.0565636344
valid mse : 26166020.660018872


train rmse : 1808.2945159911408
valid rmse : 5115.273273249326


train mae : 1021.5318249229368
valid mae : 2894.5430053188793


train mape : 0.13566561166364824
valid mape : 0.35490618709467375


train r2_score : 0.9780239492538848
valid r2_score : 0.8253389669377773


In [128]:
predict_test = rf.predict(x_test_dummies)

In [129]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test
})

data

,ID,predict
0,0,9043.975407
1,1,2540.663102
2,5,7722.497417
3,21,3258.296362
4,26,12727.593857
...,...,...
263,1327,12147.022323
264,1328,13154.932330
265,1329,2705.690547
266,1331,35651.259848


## 킹카운티 주거지 가격예측문제 데이터
데이터 설명 : 킹카운티 주거지 가격 예측문제 (종속변수 :price)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/y_test.csv  
데이터 출처 :https://www.kaggle.com/harlfoxem/housesalesprediction (참고, 데이터 수정)

In [130]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/kingcountyprice/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,id,date,bedrooms,bathrooms,sqft_living,sqft_lot,floors,waterfront,view,...,grade,sqft_above,sqft_basement,yr_built,yr_renovated,zipcode,lat,long,sqft_living15,sqft_lot15
0,2,8651400730,20150428T000000,3,1.00,840,5525,1.0,0,0,...,6,840,0,1969,0,98042,47.3607,-122.085,920,5330
1,3,3163600130,20150317T000000,3,1.00,1250,8000,1.0,0,0,...,7,1250,0,1956,0,98146,47.5065,-122.337,1040,6973
2,4,5045700330,20140725T000000,4,2.50,2200,6400,2.0,0,0,...,8,2200,0,2010,0,98059,47.4856,-122.156,2600,5870
3,5,1036100130,20140808T000000,3,2.50,1980,39932,2.0,0,0,...,8,1980,0,1994,0,98011,47.7433,-122.196,2610,12769
4,6,7696630080,20140506T000000,3,1.75,1690,7735,1.0,0,0,...,7,1060,630,1976,0,98001,47.3324,-122.280,1580,7503


,ID,price
0,2,191000.0
1,3,234900.0
2,4,460000.0
3,5,442000.0
4,6,197000.0


In [131]:
x_train['date'] = pd.to_datetime(x_train['date'])
x_test['date'] = pd.to_datetime(x_test['date'])

x_train["year"] = x_train["date"].dt.year
x_test["year"] = x_test["date"].dt.year

x_train["month"] = x_train["date"].dt.month
x_test["month"] = x_test["date"].dt.month

x_train["day"] = x_train["date"].dt.day
x_test["day"] = x_test["date"].dt.day

In [132]:
drop_col = ["ID", "id", "date"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["price"]

In [133]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [134]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)
print(len(X_train), len(X_valid), len(x_train_dummies))

12103 5187 17290


In [135]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [136]:
predict_train = rf.predict(X_train)
predict_valid = rf.predict(X_valid)

In [137]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
print("train mse :", mean_squared_error(Y_train, predict_train))
print("valid mse :", mean_squared_error(Y_valid, predict_valid))
print("\n")

print("train rmse :", np.sqrt(mean_squared_error(Y_train, predict_train)))
print("valid rmse :", np.sqrt(mean_squared_error(Y_valid, predict_valid)))
print("\n")

print("train mae :", mean_absolute_error(Y_train, predict_train))
print("valid mae :", mean_absolute_error(Y_valid, predict_valid))
print("\n")

print("train mape :", mean_absolute_percentage_error(Y_train, predict_train))
print("valid mape :", mean_absolute_percentage_error(Y_valid, predict_valid))
print("\n")

print("train r2_score :", r2_score(Y_train, predict_train))
print("valid r2_score :", r2_score(Y_valid, predict_valid))

train mse : 2582665845.8904243
valid mse : 19942911113.75906


train rmse : 50819.9355163938
valid rmse : 141219.37230337437


train mae : 26375.50215979509
valid mae : 72123.73808559861


train mape : 0.04932272135321542
valid mape : 0.13391769051597954


train r2_score : 0.9808096173596833
valid r2_score : 0.8691363985680141


In [138]:
predict_test = rf.predict(x_test_dummies)

In [139]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test
})

data

,ID,predict
0,0,356681.88
1,1,471041.94
2,14,414319.18
3,22,542184.50
4,26,523984.00
...,...,...
4318,21573,1718145.00
4319,21584,430369.40
4320,21585,312783.98
4321,21588,450680.34


## 대학원 입학가능성 데이터
데이터 설명 : 대학원 입학 가능성 예측 (종속변수 :Chance of Admit)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/y_test.csv  
데이터 출처 :https://www.kaggle.com/mohansacharya/graduate-admissions(참고, 데이터 수정)

In [140]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/admission/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,Serial No.,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research
0,0,67,327,114,3,3.0,3.0,9.02,0
1,1,112,321,109,4,4.0,4.0,8.68,1
2,2,495,301,99,3,2.5,2.0,8.45,1
3,3,356,317,106,2,2.0,3.5,8.12,0
4,4,250,321,111,3,3.5,4.0,8.83,1


,ID,Chance of Admit
0,0,0.61
1,1,0.69
2,2,0.68
3,3,0.73
4,4,0.77


In [141]:
drop_col = ["ID", "Serial No."]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["Chance of Admit"]

In [142]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [143]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)
print(len(X_train), len(X_valid), len(x_train_dummies))

280 120 400


In [144]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [145]:
predict_train = rf.predict(X_train)
predict_valid = rf.predict(X_valid)

In [146]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
print("train mse :", mean_squared_error(Y_train, predict_train))
print("valid mse :", mean_squared_error(Y_valid, predict_valid))
print("\n")

print("train rmse :", np.sqrt(mean_squared_error(Y_train, predict_train)))
print("valid rmse :", np.sqrt(mean_squared_error(Y_valid, predict_valid)))
print("\n")

print("train mae :", mean_absolute_error(Y_train, predict_train))
print("valid mae :", mean_absolute_error(Y_valid, predict_valid))
print("\n")

print("train mape :", mean_absolute_percentage_error(Y_train, predict_train))
print("valid mape :", mean_absolute_percentage_error(Y_valid, predict_valid))
print("\n")

print("train r2_score :", r2_score(Y_train, predict_train))
print("valid r2_score :", r2_score(Y_valid, predict_valid))

train mse : 0.0005858450714285725
valid mse : 0.004448170416666664


train rmse : 0.024204236642137104
valid rmse : 0.06669460560395168


train mae : 0.017392142857142894
valid mae : 0.0482641666666667


train mape : 0.02812003229358026
valid mape : 0.07367360569863518


train r2_score : 0.9716851696762848
valid r2_score : 0.7539033050764106


In [147]:
predict_test = rf.predict(x_test_dummies)

In [148]:
data = pd.DataFrame({
    "predict" : predict_test
})

data

,predict
0,0.7013
1,0.7191
2,0.8477
3,0.6009
4,0.6763
...,...
95,0.5601
96,0.9200
97,0.6481
98,0.9034


## 레드 와인 퀄리티 예측 데이터
데이터 설명 : 레드 와인 퀄리티 예측문제 (종속변수 :quality)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/y_test.csv  
데이터 출처 :https://www.kaggle.com/uciml/red-wine-quality-cortez-et-al-2009(참고, 데이터 수정)

In [149]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/redwine/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol
0,1,10.6,0.44,0.68,4.1,0.114,6.0,24.0,0.99700,3.06,0.66,13.4
1,2,7.0,0.60,0.30,4.5,0.068,20.0,110.0,0.99914,3.30,1.17,10.2
2,3,8.0,0.43,0.36,2.3,0.075,10.0,48.0,0.99760,3.34,0.46,9.4
3,4,7.9,0.53,0.24,2.0,0.072,15.0,105.0,0.99600,3.27,0.54,9.4
4,5,8.0,0.45,0.23,2.2,0.094,16.0,29.0,0.99620,3.21,0.49,10.2


,ID,quality
0,1,6
1,2,5
2,3,5
3,4,6
4,5,6


In [150]:
drop_col =  ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["quality"]

In [151]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies[x_train_dummies.columns]

In [152]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)
print(len(X_train), len(X_valid), len(x_train_dummies))

895 384 1279


In [153]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [154]:
predict_train = rf.predict(X_train)
predict_valid = rf.predict(X_valid)

In [155]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
print("train mse :", mean_squared_error(Y_train, predict_train))
print("valid mse :", mean_squared_error(Y_valid, predict_valid))
print("\n")

print("train rmse :", np.sqrt(mean_squared_error(Y_train, predict_train)))
print("valid rmse :", np.sqrt(mean_squared_error(Y_valid, predict_valid)))
print("\n")

print("train mae :", mean_absolute_error(Y_train, predict_train))
print("valid mae :", mean_absolute_error(Y_valid, predict_valid))
print("\n")

print("train mape :", mean_absolute_percentage_error(Y_train, predict_train))
print("valid mape :", mean_absolute_percentage_error(Y_valid, predict_valid))
print("\n")

print("train r2_score :", r2_score(Y_train, predict_train))
print("valid r2_score :", r2_score(Y_valid, predict_valid))

train mse : 0.044542681564245803
valid mse : 0.41343151041666665


train rmse : 0.21105137186061076
valid rmse : 0.6429863998691315


train mae : 0.15058100558659215
valid mae : 0.4702864583333333


train mape : 0.028038002128225586
valid mape : 0.08414524429563491


train r2_score : 0.9301715942806147
valid r2_score : 0.39614330203256864


In [156]:
predict_test = rf.predict(x_test_dummies)

In [157]:
data = pd.DataFrame({
    "predict" : predict_test
})

data

,predict
0,5.76
1,6.62
2,5.05
3,3.54
4,4.91
...,...
315,5.08
316,5.25
317,4.93
318,5.06


## 현대 차량 가격 분류문제 데이터
데이터 설명 : 현대 차량가격 분류문제 (종속변수 :price)  
x_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_train.csv  
y_train: https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/y_train.csv  
x_test: https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_test.csv  
x_label(평가용) : https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/y_test.csv  
데이터 출처 :https://www.kaggle.com/mysarahmadbhat/hyundai-used-car-listing(참고, 데이터 수정)

In [158]:
import pandas as pd

x_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_train.csv")
y_train = pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/y_train.csv")
x_test= pd.read_csv("https://raw.githubusercontent.com/Datamanim/datarepo/main/hyundai/x_test.csv")

display(x_train.head())
display(y_train.head())

,ID,model,year,transmission,mileage,fuelType,tax(£),mpg,engineSize
0,0,I30,2019,Manual,21,Petrol,150,34.0,2.0
1,1,Santa Fe,2018,Semi-Auto,10500,Diesel,145,39.8,2.2
2,2,Tucson,2017,Manual,29968,Diesel,30,61.7,1.7
3,3,Kona,2018,Manual,27317,Petrol,145,52.3,1.0
4,4,Tucson,2018,Semi-Auto,31459,Diesel,145,57.7,1.7


,ID,price
0,0,23995
1,1,28490
2,2,13251
3,3,14990
4,4,17591


In [159]:
drop_col = ["ID"]
x_train_drop = x_train.drop(columns = drop_col)
x_test_drop = x_test.drop(columns = drop_col)
y = y_train["price"]

In [160]:
x_train_dummies = pd.get_dummies(x_train_drop)
x_test_dummies = pd.get_dummies(x_test_drop)
x_test_dummies = x_test_dummies.reindex(columns = x_train_dummies.columns, fill_value = 0)

In [161]:
from sklearn.model_selection import train_test_split
X_train, X_valid, Y_train, Y_valid = train_test_split(x_train_dummies, y, test_size = 0.3, random_state = 42)
print(len(X_train), len(X_valid), len(x_train_dummies))

2721 1167 3888


In [162]:
from sklearn.ensemble import RandomForestRegressor
rf = RandomForestRegressor(random_state = 23)
rf.fit(X_train, Y_train)

,n_estimators,100
,criterion,'squared_error'
,max_depth,None
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,1.0
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,bootstrap,True
,oob_score,False


In [163]:
predict_train = rf.predict(X_train)
predict_valid = rf.predict(X_valid)

In [164]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error, r2_score
print("train mse :", mean_squared_error(Y_train, predict_train))
print("valid mse :", mean_squared_error(Y_valid, predict_valid))
print("\n")

print("train rmse :", np.sqrt(mean_squared_error(Y_train, predict_train)))
print("valid rmse :", np.sqrt(mean_squared_error(Y_valid, predict_valid)))
print("\n")

print("train mae :", mean_absolute_error(Y_train, predict_train))
print("valid mae :", mean_absolute_error(Y_valid, predict_valid))
print("\n")

print("train mape :", mean_absolute_percentage_error(Y_train, predict_train))
print("valid mape :", mean_absolute_percentage_error(Y_valid, predict_valid))
print("\n")

print("train r2_score :", r2_score(Y_train, predict_train))
print("valid r2_score :", r2_score(Y_valid, predict_valid))

train mse : 231650.89844824854
valid mse : 1545998.8273950836


train rmse : 481.30125539857937
valid rmse : 1243.3820118511783


train mae : 322.05876511445393
valid mae : 847.0117319383938


train mape : 0.027224210729746626
valid mape : 0.06865018129441026


train r2_score : 0.9930921371120084
valid r2_score : 0.9580707233859072


In [165]:
predict_test = rf.predict(x_test_dummies)

In [166]:
data = pd.DataFrame({
    "ID" : x_test["ID"],
    "predict" : predict_test
})

data

,ID,predict
0,21,8642.29
1,28,5132.18
2,33,35668.62
3,38,13522.61
4,45,8386.48
...,...,...
967,4833,1920.16
968,4840,18561.07
969,4842,6102.46
970,4843,8575.82
